In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

In [12]:
import os
from langchain_groq import ChatGroq 

os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(model="llama-3.1-8b-instant")

In [13]:
from typing import TypedDict 
from typing_extensions import TypedDict

class JokeState(TypedDict):
    
    topic: str
    joke: str
    explanation: str

In [18]:
def generate_joke(state: JokeState):

    prompt = f"Generate a short joke about {state['topic']} in one line."

    response = llm.invoke([HumanMessage(content=prompt)])

    joke = response.content.replace("\n", " ")

    return {"joke": joke}

In [19]:
def generate_explanation(state: JokeState):

    prompt = f"Explain this joke in one line: {state['joke']}"

    response = llm.invoke([HumanMessage(content=prompt)])

    explanation = response.content.replace("\n", " ")

    return {"explanation": explanation}

In [20]:
graph = StateGraph(JokeState)
    
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)
    
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)
    
checkpointer = InMemorySaver()
    
workflow = graph.compile(checkpointer=checkpointer)

In [21]:
config1 = {"configurable": {"thread_id": "1"}}
result = workflow.invoke({"topic":"pizza"}, config=config1)
print(result)

{'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling crusty.', 'explanation': 'This joke is a play on words, using the phrase "feeling crusty" to refer to both the pizza\'s edible crust and the common expression for feeling irritable or grumpy.'}


In [22]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling crusty.', 'explanation': 'This joke is a play on words, using the phrase "feeling crusty" to refer to both the pizza\'s edible crust and the common expression for feeling irritable or grumpy.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11e245-cb5b-6bb6-8002-5de163c3fafe'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-12T15:01:34.826591+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11e245-c938-6682-8001-4623ce4a8a0d'}}, tasks=(), interrupts=())

In [ ]:
list(workflow.get_state_history(config1)) 

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling crusty.', 'explanation': 'This joke is a play on words, using the phrase "feeling crusty" to refer to both the pizza\'s edible crust and the common expression for feeling irritable or grumpy.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11e245-cb5b-6bb6-8002-5de163c3fafe'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-03-12T15:01:34.826591+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11e245-c938-6682-8001-4623ce4a8a0d'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling crusty.'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f11e245-c938-6682-8001-4623ce4a8a0d'}}, metadata={'source': 'loop', 'step': 1

In [ ]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)  

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy? It was feeling a little "twisted."',
 'explanation': 'The joke is a play on words, where "twisted" has a double meaning referring to both the twisted shape of spaghetti and that the spaghetti is feeling emotionally disturbed.'}